<a href="https://colab.research.google.com/github/ShachiPradhan/openFDA/blob/main/openFDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests

BASE = "https://api.fda.gov/drug/event.json"

r = requests.get(BASE, params={
    "search": 'patient.reaction.reactionmeddrapt:"headache"',
    "limit": 5,
})
data = r.json()
print(data["meta"]["results"]["total"])

629244


In [ ]:
import json
print(json.dumps(data["results"][0], indent=2))

{
  "safetyreportversion": "1",
  "safetyreportid": "10003300",
  "primarysourcecountry": "US",
  "transmissiondateformat": "102",
  "transmissiondate": "20141002",
  "reporttype": "1",
  "serious": "1",
  "seriousnessdisabling": "1",
  "receivedateformat": "102",
  "receivedate": "20140306",
  "receiptdateformat": "102",
  "receiptdate": "20140306",
  "fulfillexpeditecriteria": "2",
  "companynumb": "1289378",
  "duplicate": "1",
  "reportduplicate": {
    "duplicatesource": "GENENTECH",
    "duplicatenumb": "1289378"
  },
  "primarysource": {
    "reportercountry": "US",
    "qualification": "5"
  },
  "sender": {
    "sendertype": "2",
    "senderorganization": "FDA-Public Use"
  },
  "receiver": {
    "receivertype": "6",
    "receiverorganization": "FDA"
  },
  "patient": {
    "patientonsetage": "77",
    "patientonsetageunit": "801",
    "patientsex": "2",
    "reaction": [
      {
        "reactionmeddraversionpt": "17.0",
        "reactionmeddrapt": "Vomiting"
      },
      {

In [ ]:
r = requests.get(BASE, params={
    "search": 'patient.drug.medicinalproduct:"lisinopril"',
    "limit": 3,
})
print(json.dumps(r.json()["results"][0], indent=2))

{
  "safetyreportversion": "1",
  "safetyreportid": "10003305",
  "primarysourcecountry": "US",
  "occurcountry": "US",
  "transmissiondateformat": "102",
  "transmissiondate": "20141002",
  "reporttype": "1",
  "serious": "2",
  "receivedateformat": "102",
  "receivedate": "20140312",
  "receiptdateformat": "102",
  "receiptdate": "20140312",
  "fulfillexpeditecriteria": "2",
  "companynumb": "US-PFIZER INC-2014069067",
  "duplicate": "1",
  "reportduplicate": {
    "duplicatesource": "PFIZER",
    "duplicatenumb": "US-PFIZER INC-2014069067"
  },
  "primarysource": {
    "reportercountry": "US",
    "qualification": "1"
  },
  "sender": {
    "sendertype": "2",
    "senderorganization": "FDA-Public Use"
  },
  "receiver": {
    "receivertype": "6",
    "receiverorganization": "FDA"
  },
  "patient": {
    "patientonsetage": "48",
    "patientonsetageunit": "801",
    "patientsex": "2",
    "reaction": [
      {
        "reactionmeddraversionpt": "17.0",
        "reactionmeddrapt": "Dr

In [ ]:
API_KEY = "fvoxNcbzYreJn9wa858Ohd3nKgfs9iYmtOgmb8Dx"

r = requests.get(BASE, params={
    "api_key": API_KEY,
    "search": 'patient.drug.medicinalproduct:"aspirin"',
    "limit": 100,
})

In [ ]:
r = requests.get(BASE, params={
    "search": 'patient.drug.medicinalproduct:"aspirin"',
    "limit": 1,
})
print(r.json()["meta"]["results"]["total"])

617924


In [ ]:
import requests
import time
import json

BASE = "https://api.fda.gov/drug/event.json"
API_KEY = "fvoxNcbzYreJn9wa858Ohd3nKgfs9iYmtOgmb8Dx"  # or delete this line if you don't have one

def fetch_page(search, limit=100, skip=0):
    params = {"search": search, "limit": limit, "skip": skip}
    if API_KEY:
        params["api_key"] = API_KEY
    r = requests.get(BASE, params=params)
    if r.status_code == 404:
        return None  # no more results
    r.raise_for_status()
    return r.json()

def fetch_all(search, max_records=2000, cache_file="raw_data.json"):
    all_results = []
    skip = 0
    limit = 100
    while len(all_results) < max_records:
        data = fetch_page(search, limit=limit, skip=skip)
        if data is None or "results" not in data:
            break
        all_results.extend(data["results"])
        print(f"Fetched {len(all_results)} so far...")
        skip += limit
        time.sleep(0.5)  # be polite to the server
    # Cache to disk so you don't re-fetch while debugging
    with open(cache_file, "w") as f:
        json.dump(all_results, f)
    return all_results

In [ ]:
reports = fetch_all('patient.drug.medicinalproduct:"aspirin"', max_records=2000)
print(f"Got {len(reports)} reports")

Fetched 100 so far...
Fetched 200 so far...
Fetched 300 so far...
Fetched 400 so far...
Fetched 500 so far...
Fetched 600 so far...
Fetched 700 so far...
Fetched 800 so far...
Fetched 900 so far...
Fetched 1000 so far...
Fetched 1100 so far...
Fetched 1200 so far...
Fetched 1300 so far...
Fetched 1400 so far...
Fetched 1500 so far...
Fetched 1600 so far...
Fetched 1700 so far...
Fetched 1800 so far...
Fetched 1900 so far...
Fetched 2000 so far...
Got 2000 reports


In [ ]:
rows = []

for report in reports:
    report_id = report.get("safetyreportid", "unknown")
    patient = report.get("patient", {})

    drugs = patient.get("drug", []) or []
    reactions = patient.get("reaction", []) or []

    drug_names = [d.get("medicinalproduct", "UNKNOWN") for d in drugs if d]
    reaction_names = [r.get("reactionmeddrapt", "UNKNOWN") for r in reactions if r]

    for drug in drug_names:
        for reaction in reaction_names:
            rows.append({
                "report_id": report_id,
                "drug": drug,
                "reaction": reaction,
            })

print(f"Made {len(rows)} rows from {len(reports)} reports")

Made 109255 rows from 2000 reports


In [ ]:
import pandas as pd
df = pd.DataFrame(rows)
print(df.head())
print(df.shape)

  report_id                    drug               reaction
0  10003304     DOXYCYCLINE HYCLATE  Drug hypersensitivity
1  10003304  TRAMADOL HYDROCHLORIDE  Drug hypersensitivity
2  10003304               OXYCONTIN  Drug hypersensitivity
3  10003304                  TALWIN  Drug hypersensitivity
4  10003304                 CODEINE  Drug hypersensitivity
(109255, 3)


In [ ]:
def compute_ror(df, drug, reaction):
    # a: has both
    a = ((df["drug"] == drug) & (df["reaction"] == reaction)).sum()

    # b: has drug, not reaction
    b = ((df["drug"] == drug) & (df["reaction"] != reaction)).sum()

    # c: has reaction, not drug
    c = ((df["drug"] != drug) & (df["reaction"] == reaction)).sum()

    # d: has neither
    d = ((df["drug"] != drug) & (df["reaction"] != reaction)).sum()

    if b == 0 or c == 0:
        return None  # avoid division by zero

    ror = (a * d) / (b * c)
    return {"drug": drug, "reaction": reaction, "a": a, "b": b, "c": c, "d": d, "ror": ror}

print(compute_ror(df, "ASPIRIN", "HEADACHE"))

None


In [ ]:
def compute_all_rors(df):
    N = len(df)
    drug_counts = df["drug"].value_counts()          # how often each drug appears
    reaction_counts = df["reaction"].value_counts()  # how often each reaction appears
    pair_counts = df.groupby(["drug", "reaction"]).size()  # how often each pair appears

    results = []
    for (drug, reaction), a in pair_counts.items():
        drug_total = drug_counts[drug]
        reaction_total = reaction_counts[reaction]

        b = drug_total - a                        # has drug, not reaction
        c = reaction_total - a                    # has reaction, not drug
        d = N - drug_total - reaction_total + a   # has neither

        if b == 0 or c == 0:
            continue  # avoid divide-by-zero

        ror = (a * d) / (b * c)
        results.append({
            "drug": drug,
            "reaction": reaction,
            "a": a, "b": b, "c": c, "d": d,
            "ror": ror,
        })

    return pd.DataFrame(results).sort_values("ror", ascending=False)

In [ ]:
ror_df = compute_all_rors(df)
print(ror_df.head(20))

                                                    drug  \
23964                  ESTRADIOL TRANSDERMAL SYSTEM, USP   
48178                     ORGARAN INTRAVENOUS 1250 UNITS   
50062                                         PERSANTINE   
20                                           325 ASPIRIN   
51357                PRADAXA 150 MG BOEHRINGER INGELHEIM   
19917                                          DALMADORM   
7363                                  ASPIRIN /00002701/   
23646  EQUATE ASPIRIN 250MG, ACETAMINOPHEN 250MG, CAF...   
15146            CENTRUM MULTIPLE VITAMINS WITH MINERALS   
32157                                 INHALER FOR ASTHMA   
30799                       HYDROCHLOROTHIAZIDE/LOSARTAN   
61273                     TAMOXIFEN CITRATE TABLETS, USP   
61272                     TAMOXIFEN CITRATE TABLETS, USP   
57519                                       SARGRAMOSTIM   
20647  DEXTROMETHORPHAN/DOXYLAMINE/PARACETAMOL/PSEUDO...   
45536                                   

In [ ]:
echo "# openFDA" >> README.md
git init
git add README.md
git commit -m "first commit"
git branch -M main
git remote add origin https://github.com/ShachiPradhan/openFDA.git
git push -u origin main